# DrishtiSR — Day 1 baseline

**SIH26142.** One Kaggle session, CPU only, that answers a single question:
*is there a benchmarked bicubic floor on paper by the end of today?* If there
is not, the team switches problem statement — so every number below has to come
out of a run, not out of a memory.

This notebook contains **no logic**. Every cell either sets a variable or calls
a function from `src/`, as `AGENTS.md` requires. The behaviour behind each call
lives in `src/utils/kaggle_session.py`, where it is version-controlled and
tested on the local CPU box; if you want to change what a stage does, change
the entry point under `scripts/` or the config, push, and re-run cell 1.

## Before you start

1. **Attach the data.** Sidebar → *+ Add Input* → the dataset named by
   `paths.kaggle_dataset_dir` (`drishtisr-sen2naipv2-cache`).
2. **Internet on.** Sidebar → *Settings* → Internet. The clone, the pip
   install and the SEN2NAIPv2 catalog all need it.
3. **Accelerator: None.** Day 1 is interpolation and metrics. A GPU session
   here spends the weekly 30-hour budget on arithmetic a CPU does fine.

## How to run it

Cells top to bottom. **Cell 1 is the stop point** — read its output before
running anything else. Each stage prints its start time and how long it took,
so the saved output says what fitted in the session and what did not.

## If stage 1 fails

The failure prints the fallback recipe: change `DATASET = None` to
`DATASET = "worldstrat"` in the configuration cell and re-run from there. That
is the whole change — every stage takes the override through the same
`--set dataset.name=...`.

## Two things that will bite you

- **Re-running cell 1 after pushing new code does not reload `src/`.** Python
  caches imported modules. Restart the kernel first (*Run → Restart & clear
  cell outputs*), then run cell 1.
- **This notebook is hand-maintained**, unlike the headless job notebooks under
  `outputs/kaggle_kernels/`, which `scripts/kaggle_run.py` generates from
  `notebooks/templates/kaggle_job.py` and overwrites on every push. Edit this
  file directly; do not edit those.

In [ ]:
# =============================================================================
# CELL 1 — ENVIRONMENT CHECK.  Read the output. If anything surprises you, stop.
# =============================================================================
#
# Three things happen here, in this order, and the order matters: the repo has to
# be on disk before `src/` can be imported, and `src.utils.kaggle_session` has to
# be imported before it can install the omegaconf that `environment_report` then
# needs. That is why this module imports torch and omegaconf inside the function
# rather than at the top of the file.
#
# The clone is idempotent: re-running this cell after `git push` fetches and
# checks out the new commit rather than failing on an existing directory. It does
# NOT reload already-imported `src/` modules — restart the kernel for that.
import os
import subprocess
import sys

REPO_URL = "https://github.com/shyam0github/DrishtiSR.git"  # must clone anonymously
GIT_REF = "main"
WORKDIR = "/kaggle/working/DrishtiSR"  # under /kaggle/working, so outputs survive

# Only what the Kaggle image lacks. Pins match requirements.txt. Deliberately NOT
# `pip install -r requirements.txt`: that pins numpy/pandas/scipy/pillow, and
# installing those pins here downgrades packages the preinstalled torch was built
# against.
PIP_PACKAGES = [
    "omegaconf==2.3.0",  # config loading, every script
    "tacoreader==2.1.0",  # SEN2NAIPv2 catalog
    "lpips==0.1.4",  # perceptual metric in the baseline table
    "satalign==0.1.17",  # opensr-test's registration backend
    "mpltern==1.0.5",  # opensr-test's ternary plots
]

# opensr-test goes in on its own, with --no-deps. MEASURED locally: its full
# dependency set pulls open-clip-torch and openai-clip, needed only for the
# `clip` correctness distance this configuration does not use, and installing
# them moves numpy past what the pinned torch and scipy accept. See "Environment,
# pinned by hand" in reports/day1_gate.md. Consequence, and it is recorded in the
# report rather than hidden: the `clip` distance is unavailable; `nd` (the
# default, and what the numbers use) and `lpips` are not affected.
PIP_NO_DEPS = ["opensr-test==1.3.3"]

if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", WORKDIR, "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", WORKDIR, "log", "-1", "--oneline"], check=True)

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)

from src.utils.kaggle_session import (  # noqa: E402
    archive_outputs,
    dataset_fallback_hint,
    environment_report,
    inventory_outputs,
    pip_install,
    run_stage,
    show_markdown,
    stage_args,
)

pip_install(PIP_PACKAGES, note="(the Kaggle image already has the rest)")
pip_install(PIP_NO_DEPS, extra_args=["--no-deps"], note="(see the note above)")

ENVIRONMENT = environment_report()

## Configuration

The only cell you edit. `DATASET = "worldstrat"` is the Day 1 fallback; leave
it `None` to use whatever `dataset.name` the config says (`sen2naipv2`).

`SMOKE = True` puts every stage on the synthetic stub — no network, no mounted
data, a couple of minutes end to end. Run it once that way to prove the wiring
before spending a session on the real thing. **Smoke numbers are not results.**

In [ ]:
# =============================================================================
# CELL 2 — CONFIGURATION.  The one cell to edit.
# =============================================================================
CONFIG = "configs/base.yaml"

# THE ONE EDIT. None = use the config's own dataset.name. "worldstrat" = the
# fallback, taken when stage 1 fails for a reason specific to SEN2NAIPv2.
DATASET = None

# True = synthetic stub, ~2 minutes, wiring check only. Not a result.
SMOKE = False

# Stage 1. --index-only skips the download and indexes what is already cached,
# which is right whenever the cache dataset is attached: the mount is read-only
# and a re-download is ~8.7 hours of session time. Drop it only if you are
# deliberately fetching into /kaggle/temp.
PREPARE_ARGS = ["--index-only"]

# Stage 2. Pairs sampled for the co-registration audit.
N_ALIGNMENT_PAIRS = 50

# Stage 3. "all" evaluates every registered baseline; bicubic is the floor the
# report quotes and nearest is the sanity check underneath it.
BASELINE = "all"

# Stage 4. opensr-test scores one sample at a time and registers each twice, so
# budget seconds per sample on CPU. 0 means the whole validation split.
N_OPENSR_SAMPLES = 200

# Stage 5. Methods tabulated in the gate, and where it is written. Stage 4 runs
# one step per method here — keep the two lists in step.
GATE_METHODS = ["bicubic", "nearest"]
GATE_OUT = "reports/day1_gate.md"

# Everything worth downloading, and the single archive it is packed into. The
# archive sits OUTSIDE the directories it packs, directly under /kaggle/working.
ARCHIVE_DIRS = ["outputs", "reports"]
# Large by construction, and never a result: the sample cache and the Kaggle
# upload staging copies. On Kaggle the cache lives in /kaggle/temp or on a
# read-only mount, so this normally excludes nothing there -- it is listed
# because running these cells against a local checkout otherwise packs several
# gigabytes into the zip.
ARCHIVE_SKIP = [
    "outputs/cache",
    "outputs/kaggle_staging",
    "outputs/kaggle_staging_dryrun",
    "outputs/kaggle",
    "outputs/kaggle_kernels",
]
ARCHIVE = "/kaggle/working/drishtisr_day1_outputs.zip"

COMMON = stage_args(config=CONFIG, dataset=DATASET, smoke=SMOKE)

## Stage 1 / 5 — prepare data

Indexes the cached pairs into `outputs/manifest_<dataset>.csv`, then cuts the
geographic train/val/test split into `outputs/splits_<dataset>.csv`. Both are
one stage because the split is meaningless against a different index.

**This is the cell with a fallback.** If it fails, read the traceback, then the
recipe printed underneath it.

**Why this derives the two CSVs instead of staging them.** The headless jobs
call `stage_supporting_files()`, which copies `manifest_<dataset>.csv` and
`splits_<dataset>.csv` out of the mounted dataset so the split is read rather
than recomputed. This notebook derives them instead, for two reasons: the
WorldStrat fallback has no mount carrying them, and neither does `--smoke`, so
staging would make both of those paths abort. The consequence is worth knowing:
the split this notebook cuts is deterministic (seeded from `cfg.seed`, verified
for geographic separation) but it is cut *here*, so quote its numbers against
the gate this same session produced, not against a baseline measured on a split
that came from somewhere else.

In [ ]:
# =============================================================================
# CELL 3 — STAGE 1/5: prepare data.  Failure here prints the fallback recipe.
# =============================================================================
run_stage(
    "1/5  prepare data",
    steps=[
        # No --summary-out: the index summary is printed to stdout, which the
        # saved kernel output already keeps, and the synthetic stub cannot
        # produce one -- passing it would make --smoke fail on the stub for a
        # reason that has nothing to do with the wiring being checked.
        ("scripts/prepare_data.py", [*COMMON, *PREPARE_ARGS]),
        ("scripts/make_splits.py", COMMON),
    ],
    on_failure=dataset_fallback_hint("worldstrat"),
)

## Stage 2 / 5 — alignment QA

Measures LR/HR co-registration and returns PASS / WARN / FAIL. A sub-pixel
misalignment caps every PSNR below it, so this runs before the baseline rather
than after it: if the pairs are not aligned, the floor measured against them is
not the floor.

Writes `outputs/metrics/alignment_report.json` and the shift figures.

In [ ]:
# =============================================================================
# CELL 4 — STAGE 2/5: alignment QA
# =============================================================================
run_stage(
    "2/5  alignment QA",
    steps=[("scripts/qa_alignment.py", [*COMMON, "--n-pairs", N_ALIGNMENT_PAIRS])],
)

## Stage 3 / 5 — bicubic baseline

The number every later result is quoted against, produced by exactly the
`Evaluator` that will score the model. Threads are pinned to
`runtime.num_threads` (6, the deployment target), so `sr_time_ms` is measured
under the conditions the INT8 ONNX export will be benchmarked under.

Writes `outputs/metrics/baseline_<method>.{csv,json}` and the qualitative SAM
panel.

In [ ]:
# =============================================================================
# CELL 5 — STAGE 3/5: bicubic baseline (and the nearest-neighbour floor)
# =============================================================================
run_stage(
    "3/5  bicubic baseline",
    steps=[("scripts/run_baseline.py", [*COMMON, "--baseline", BASELINE])],
)

## Stage 4 / 5 — opensr-test

The independent check. Our own metrics say how close the output is; opensr-test
(Aybar et al., IEEE JSTARS 2024) says whether the detail a super-resolver added
is *actually in the reference*. Bicubic invents nothing, so its correctness
figures are the arithmetic floor — the number that makes a learned model's
hallucination score legible later.

**The slow stage.** Seconds per sample, one at a time, two registrations each.
Lower `N_OPENSR_SAMPLES` if the session clock is tight.

In [ ]:
# =============================================================================
# CELL 6 — STAGE 4/5: opensr-test.  One step per entry in GATE_METHODS.
# =============================================================================
run_stage(
    "4/5  opensr-test",
    steps=[
        (
            "scripts/run_opensr_test.py",
            [*COMMON, "--baseline", "bicubic", "--n-samples", N_OPENSR_SAMPLES],
        ),
        (
            "scripts/run_opensr_test.py",
            [*COMMON, "--baseline", "nearest", "--n-samples", N_OPENSR_SAMPLES],
        ),
    ],
)

## Stage 5 / 5 — the Day 1 gate

Reads the JSON and CSV the four stages above wrote and renders
`reports/day1_gate.md`. Prose is templated; **numbers are read from files**, so
nothing in the report can be a figure somebody retyped out of a log. A missing
input is named, along with the command that produces it, rather than leaving a
hole in the report.

In [ ]:
# =============================================================================
# CELL 7 — STAGE 5/5: render the gate, then put it on screen
# =============================================================================
run_stage(
    "5/5  Day 1 gate",
    steps=[
        ("scripts/make_day1_gate.py", [*COMMON, "--methods", *GATE_METHODS, "--out", GATE_OUT]),
    ],
)

GATE = show_markdown(GATE_OUT)

## Collect the results

Everything the stages wrote lives under `/kaggle/working/DrishtiSR/`, so it is
saved as this kernel's output. The inventory says what is there; the archive
makes it one download instead of several hundred.

Download it from the **Output** panel on the right of the session, or with
`python scripts/kaggle_run.py fetch` if this ran headless.

In [ ]:
# =============================================================================
# CELL 8 — what was produced, and one zip to take it home
# =============================================================================
inventory_outputs(ARCHIVE_DIRS)

archive_outputs(ARCHIVE_DIRS, ARCHIVE, skip=ARCHIVE_SKIP)